In [1]:
import pandas as pd
import sqlite3

df = pd.read_csv("../data/processed/saas_customers_cleaned.csv",
                  parse_dates=["Subscription_Date", "Renewal_Date", "Cancellation_Date"])
df["Revenue"] = df["Monthly_Fee"] * (1 - df["Discount_Percent"] / 100)

conn = sqlite3.connect("../data/saas_analytics.db")
df.to_sql("customers", conn, if_exists="replace", index=False)

print("Table loaded. Row count check:")
result = pd.read_sql("SELECT COUNT(*) as total_rows FROM customers", conn)
print(result)


Table loaded. Row count check:
   total_rows
0       35000


In [2]:
# Q1: List all Enterprise plan customers
q1 = """
SELECT Customer_ID, Country, Monthly_Fee, Customer_Status
FROM customers
WHERE Subscription_Plan = 'Enterprise'
LIMIT 10;
"""
print(pd.read_sql(q1, conn))

  Customer_ID    Country  Monthly_Fee Customer_Status
0  CUST100019        USA       241.48          Active
1  CUST100040        USA       249.07          Active
2  CUST100055  Singapore       276.31          Active
3  CUST100061        USA       284.08          Active
4  CUST100078    Germany       233.96          Active
5  CUST100079     France       233.36          Active
6  CUST100092      India       273.25          Active
7  CUST100104  Australia       285.58          Active
8  CUST100112        UAE       247.00          Active
9  CUST100125        UAE       280.35          Active


In [3]:
# Q2: Find all churned customers with high monthly fees (over $100)
q2 = """
SELECT Customer_ID, Subscription_Plan, Monthly_Fee, Customer_Satisfaction
FROM customers
WHERE Customer_Status = 'Churned' AND Monthly_Fee > 100
LIMIT 10;
"""
print(pd.read_sql(q2, conn))

  Customer_ID Subscription_Plan  Monthly_Fee  Customer_Satisfaction
0  CUST100021           Premium       109.81                    5.6
1  CUST100169           Premium       104.49                    5.6
2  CUST100200           Premium       102.00                    5.2
3  CUST100276           Premium       102.39                    5.2
4  CUST100409           Premium       105.53                    6.5
5  CUST100439        Enterprise       276.12                    6.2
6  CUST100442           Premium       105.91                    6.3
7  CUST100594           Premium       112.94                    5.6
8  CUST100643        Enterprise       234.48                    6.2
9  CUST100678        Enterprise       259.11                    5.6


In [4]:
# Q3: Customers from USA or India using the Premium plan
q3 = """
SELECT Customer_ID, Country, Subscription_Plan, Monthly_Fee
FROM customers
WHERE Country IN ('USA', 'India') AND Subscription_Plan = 'Premium'
LIMIT 10;
"""
print(pd.read_sql(q3, conn))

  Customer_ID Country Subscription_Plan  Monthly_Fee
0  CUST100001   India           Premium       110.60
1  CUST100012     USA           Premium        98.98
2  CUST100017     USA           Premium       110.81
3  CUST100025   India           Premium        99.07
4  CUST100029     USA           Premium        93.16
5  CUST100067     USA           Premium       104.92
6  CUST100072   India           Premium       100.84
7  CUST100077     USA           Premium       104.11
8  CUST100093     USA           Premium        97.92
9  CUST100105     USA           Premium       100.50


In [5]:
# Q4: Customers who never filed a support ticket
q4 = """
SELECT Customer_ID, Subscription_Plan, Usage_Hours_Monthly
FROM customers
WHERE Support_Tickets = 0
LIMIT 10;
"""
print(pd.read_sql(q4, conn))

  Customer_ID Subscription_Plan  Usage_Hours_Monthly
0  CUST100014             Basic                 16.3
1  CUST100036             Basic                 14.0
2  CUST100038             Basic                 11.0
3  CUST100055        Enterprise                 36.3
4  CUST100063             Basic                 13.0
5  CUST100105           Premium                 10.9
6  CUST100120             Basic                  5.6
7  CUST100138          Standard                 11.7
8  CUST100140        Enterprise                 28.0
9  CUST100141           Premium                 19.5


In [6]:
# Q5: Customers with satisfaction below 4 (at-risk / unhappy customers)
q5 = """
SELECT Customer_ID, Customer_Satisfaction, Support_Tickets, Customer_Status
FROM customers
WHERE Customer_Satisfaction < 4
ORDER BY Customer_Satisfaction ASC
LIMIT 10;
"""
print(pd.read_sql(q5, conn))

  Customer_ID  Customer_Satisfaction  Support_Tickets Customer_Status
0  CUST119998                    1.0                8         Churned
1  CUST116454                    1.9                7         Churned
2  CUST120940                    2.0                9         Churned
3  CUST104641                    2.1                4         Churned
4  CUST128566                    2.1                9         Churned
5  CUST104321                    2.2                7         Churned
6  CUST101046                    2.3                9         Churned
7  CUST105101                    2.3                6         Churned
8  CUST111462                    2.3                7         Churned
9  CUST113084                    2.3                5         Churned


In [7]:
# Q6: Count of customers per subscription plan
q6 = """
SELECT Subscription_Plan, COUNT(*) as Customer_Count
FROM customers
GROUP BY Subscription_Plan
ORDER BY Customer_Count DESC;
"""
print(pd.read_sql(q6, conn))

  Subscription_Plan  Customer_Count
0             Basic           12979
1          Standard           10794
2           Premium            7275
3        Enterprise            3952


In [8]:
# Q7: Average monthly fee and average satisfaction per plan
q7 = """
SELECT Subscription_Plan,
       ROUND(AVG(Monthly_Fee), 2) as Avg_Fee,
       ROUND(AVG(Customer_Satisfaction), 2) as Avg_Satisfaction
FROM customers
GROUP BY Subscription_Plan
ORDER BY Avg_Fee DESC;
"""
print(pd.read_sql(q7, conn))

  Subscription_Plan  Avg_Fee  Avg_Satisfaction
0        Enterprise   255.27              7.53
1           Premium   101.48              7.16
2          Standard    46.09              6.74
3             Basic    15.38              6.19


In [9]:
# Q8: Countries with more than 3000 customers (using HAVING)
q8 = """
SELECT Country, COUNT(*) as Customer_Count
FROM customers
GROUP BY Country
HAVING COUNT(*) > 3000
ORDER BY Customer_Count DESC;
"""
print(pd.read_sql(q8, conn))

  Country  Customer_Count
0     USA            9824
1   India            6315
2      UK            3465


In [10]:
# Q9: Churn rate by marketing channel
q9 = """
SELECT Marketing_Channel,
       COUNT(*) as Total_Customers,
       SUM(CASE WHEN Customer_Status = 'Churned' THEN 1 ELSE 0 END) as Churned_Count,
       ROUND(100.0 * SUM(CASE WHEN Customer_Status = 'Churned' THEN 1 ELSE 0 END) / COUNT(*), 2) as Churn_Rate_Pct
FROM customers
GROUP BY Marketing_Channel
ORDER BY Churn_Rate_Pct DESC;
"""
print(pd.read_sql(q9, conn))

  Marketing_Channel  Total_Customers  Churned_Count  Churn_Rate_Pct
0      Social Media             7115           1965           27.62
1    Organic Search             7700           2123           27.57
2          Referral             4915           1328           27.02
3          Paid Ads             8335           2237           26.84
4         Affiliate             2776            736           26.51
5    Email Campaign             4159           1076           25.87


In [11]:
# Q10: Average lifetime value by company size
q10 = """
SELECT Company_Size,
       COUNT(*) as Customer_Count,
       ROUND(AVG(Lifetime_Value), 2) as Avg_LTV
FROM customers
GROUP BY Company_Size
ORDER BY Avg_LTV DESC;
"""
print(pd.read_sql(q10, conn))

  Company_Size  Customer_Count  Avg_LTV
0   Enterprise            4482  3688.39
1       Medium           11281  1628.28
2        Small           19237   855.99


In [12]:
# Q11: Categorize customers into value tiers based on Lifetime Value
q11 = """
SELECT Customer_ID, Lifetime_Value,
    CASE
        WHEN Lifetime_Value >= 3000 THEN 'High Value'
        WHEN Lifetime_Value >= 500 THEN 'Medium Value'
        ELSE 'Low Value'
    END as Value_Tier
FROM customers
ORDER BY Lifetime_Value DESC
LIMIT 10;
"""
print(pd.read_sql(q11, conn))

  Customer_ID  Lifetime_Value  Value_Tier
0  CUST113367        13327.19  High Value
1  CUST101936        13172.00  High Value
2  CUST105294        13165.06  High Value
3  CUST111450        13038.53  High Value
4  CUST120011        12944.15  High Value
5  CUST118206        12926.75  High Value
6  CUST110345        12633.44  High Value
7  CUST131254        12577.89  High Value
8  CUST110569        12564.98  High Value
9  CUST127473        12541.45  High Value


In [13]:
# Q12: Count customers in each value tier
q12 = """
SELECT
    CASE
        WHEN Lifetime_Value >= 3000 THEN 'High Value'
        WHEN Lifetime_Value >= 500 THEN 'Medium Value'
        ELSE 'Low Value'
    END as Value_Tier,
    COUNT(*) as Customer_Count,
    ROUND(AVG(Lifetime_Value), 2) as Avg_LTV_In_Tier
FROM customers
GROUP BY Value_Tier
ORDER BY Avg_LTV_In_Tier DESC;
"""
print(pd.read_sql(q12, conn))

     Value_Tier  Customer_Count  Avg_LTV_In_Tier
0    High Value            5090          5693.81
1  Medium Value           14604          1310.36
2     Low Value           15306           212.26


In [14]:
# Q13: Top 10 customers by revenue, with a risk flag for high support tickets
q13 = """
SELECT Customer_ID, Monthly_Fee, Support_Tickets,
    CASE WHEN Support_Tickets >= 5 THEN 'At Risk' ELSE 'Healthy' END as Risk_Flag
FROM customers
WHERE Customer_Status = 'Active'
ORDER BY Monthly_Fee DESC
LIMIT 10;
"""
print(pd.read_sql(q13, conn))

  Customer_ID  Monthly_Fee  Support_Tickets Risk_Flag
0  CUST128701       286.33                1   Healthy
1  CUST118347       286.31                2   Healthy
2  CUST113800       286.29                2   Healthy
3  CUST102119       286.27                0   Healthy
4  CUST113472       286.24                3   Healthy
5  CUST107548       286.21                1   Healthy
6  CUST128295       286.20                0   Healthy
7  CUST103107       286.18                1   Healthy
8  CUST130125       286.18                2   Healthy
9  CUST133032       286.17                1   Healthy


In [15]:
# Q14: Age groups and their average satisfaction
q14 = """
SELECT
    CASE
        WHEN Age < 30 THEN 'Under 30'
        WHEN Age < 45 THEN '30-44'
        WHEN Age < 60 THEN '45-59'
        ELSE '60+'
    END as Age_Group,
    COUNT(*) as Customer_Count,
    ROUND(AVG(Customer_Satisfaction), 2) as Avg_Satisfaction
FROM customers
GROUP BY Age_Group
ORDER BY Age_Group;
"""
print(pd.read_sql(q14, conn))

  Age_Group  Customer_Count  Avg_Satisfaction
0     30-44           11197              6.73
1     45-59           10833              6.71
2       60+            4382              6.69
3  Under 30            8588              6.70


In [16]:
# Q15: Sort customers by discount given, highest first, showing plan and fee
q15 = """
SELECT Customer_ID, Subscription_Plan, Monthly_Fee, Discount_Percent
FROM customers
ORDER BY Discount_Percent DESC
LIMIT 10;
"""
print(pd.read_sql(q15, conn))

  Customer_ID Subscription_Plan  Monthly_Fee  Discount_Percent
0  CUST124270             Basic        17.10              23.8
1  CUST123360          Standard        51.65              23.6
2  CUST122511        Enterprise       251.57              23.5
3  CUST109609          Standard        41.85              23.4
4  CUST123376             Basic        16.18              22.7
5  CUST125121             Basic        16.23              22.7
6  CUST112378          Standard        40.96              22.5
7  CUST132384             Basic        14.68              22.2
8  CUST100860          Standard        41.09              21.9
9  CUST103651           Premium       110.88              21.6


In [17]:
plan_details = pd.DataFrame({
    "Subscription_Plan": ["Basic", "Standard", "Premium", "Enterprise"],
    "Max_Users_Allowed": [5, 20, 100, 1000],
    "Support_Level": ["Email Only", "Email + Chat", "Priority Support", "Dedicated Account Manager"],
    "Onboarding_Included": [0, 0, 1, 1]  # 0 = No, 1 = Yes
})

plan_details.to_sql("plan_details", conn, if_exists="replace", index=False)
print(pd.read_sql("SELECT * FROM plan_details", conn))

  Subscription_Plan  Max_Users_Allowed              Support_Level  \
0             Basic                  5                 Email Only   
1          Standard                 20               Email + Chat   
2           Premium                100           Priority Support   
3        Enterprise               1000  Dedicated Account Manager   

   Onboarding_Included  
0                    0  
1                    0  
2                    1  
3                    1  


In [18]:
# Q16: Show customers along with their plan's support level and max users
q16 = """
SELECT c.Customer_ID, c.Subscription_Plan, c.Monthly_Fee,
       p.Support_Level, p.Max_Users_Allowed
FROM customers c
JOIN plan_details p ON c.Subscription_Plan = p.Subscription_Plan
LIMIT 10;
"""
print(pd.read_sql(q16, conn))

  Customer_ID Subscription_Plan  Monthly_Fee     Support_Level  \
0  CUST100000             Basic        15.87        Email Only   
1  CUST100001           Premium       110.60  Priority Support   
2  CUST100002             Basic        15.27        Email Only   
3  CUST100003           Premium        98.04  Priority Support   
4  CUST100004          Standard        40.68      Email + Chat   
5  CUST100005          Standard        51.66      Email + Chat   
6  CUST100006           Premium        97.72  Priority Support   
7  CUST100007           Premium       110.76  Priority Support   
8  CUST100008             Basic        15.50        Email Only   
9  CUST100009           Premium        90.16  Priority Support   

   Max_Users_Allowed  
0                  5  
1                100  
2                  5  
3                100  
4                 20  
5                 20  
6                100  
7                100  
8                  5  
9                100  


In [19]:
# Q17: Count how many customers get onboarding included, broken down by plan
q17 = """
SELECT p.Subscription_Plan, p.Onboarding_Included, COUNT(*) as Customer_Count
FROM customers c
JOIN plan_details p ON c.Subscription_Plan = p.Subscription_Plan
GROUP BY p.Subscription_Plan, p.Onboarding_Included
ORDER BY Customer_Count DESC;
"""
print(pd.read_sql(q17, conn))

  Subscription_Plan  Onboarding_Included  Customer_Count
0             Basic                    0           12979
1          Standard                    0           10794
2           Premium                    1            7275
3        Enterprise                    1            3952


In [20]:
# Q18: Average satisfaction for customers who get Priority Support vs not
q18 = """
SELECT p.Support_Level, ROUND(AVG(c.Customer_Satisfaction), 2) as Avg_Satisfaction, COUNT(*) as Customer_Count
FROM customers c
JOIN plan_details p ON c.Subscription_Plan = p.Subscription_Plan
GROUP BY p.Support_Level
ORDER BY Avg_Satisfaction DESC;
"""
print(pd.read_sql(q18, conn))

               Support_Level  Avg_Satisfaction  Customer_Count
0  Dedicated Account Manager              7.53            3952
1           Priority Support              7.16            7275
2               Email + Chat              6.74           10794
3                 Email Only              6.19           12979


In [21]:
# Q19: Enterprise customers exceeding their plan's max users would be flagged here
# (illustrative - our dataset doesn't track actual user count per account, 
#  but this shows how you'd structure the check if it did)
q19 = """
SELECT c.Customer_ID, c.Subscription_Plan, p.Max_Users_Allowed
FROM customers c
JOIN plan_details p ON c.Subscription_Plan = p.Subscription_Plan
WHERE c.Subscription_Plan = 'Enterprise'
LIMIT 10;
"""
print(pd.read_sql(q19, conn))

  Customer_ID Subscription_Plan  Max_Users_Allowed
0  CUST100019        Enterprise               1000
1  CUST100040        Enterprise               1000
2  CUST100055        Enterprise               1000
3  CUST100061        Enterprise               1000
4  CUST100078        Enterprise               1000
5  CUST100079        Enterprise               1000
6  CUST100092        Enterprise               1000
7  CUST100104        Enterprise               1000
8  CUST100112        Enterprise               1000
9  CUST100125        Enterprise               1000


In [22]:
# Q20: LEFT JOIN example - show all plans, even if (hypothetically) some had zero customers
q20 = """
SELECT p.Subscription_Plan, p.Support_Level, COUNT(c.Customer_ID) as Customer_Count
FROM plan_details p
LEFT JOIN customers c ON p.Subscription_Plan = c.Subscription_Plan
GROUP BY p.Subscription_Plan, p.Support_Level
ORDER BY Customer_Count DESC;
"""
print(pd.read_sql(q20, conn))

  Subscription_Plan              Support_Level  Customer_Count
0             Basic                 Email Only           12979
1          Standard               Email + Chat           10794
2           Premium           Priority Support            7275
3        Enterprise  Dedicated Account Manager            3952


In [23]:
# Q21: Customers whose Monthly_Fee is above the overall average
q21 = """
SELECT Customer_ID, Subscription_Plan, Monthly_Fee
FROM customers
WHERE Monthly_Fee > (SELECT AVG(Monthly_Fee) FROM customers)
ORDER BY Monthly_Fee DESC
LIMIT 10;
"""
print(pd.read_sql(q21, conn))

  Customer_ID Subscription_Plan  Monthly_Fee
0  CUST128701        Enterprise       286.33
1  CUST118347        Enterprise       286.31
2  CUST113800        Enterprise       286.29
3  CUST102119        Enterprise       286.27
4  CUST113472        Enterprise       286.24
5  CUST107548        Enterprise       286.21
6  CUST128295        Enterprise       286.20
7  CUST103107        Enterprise       286.18
8  CUST130125        Enterprise       286.18
9  CUST133032        Enterprise       286.17


In [24]:
# Q22: Countries with above-average churn rate
q22 = """
SELECT Country,
       ROUND(100.0 * SUM(CASE WHEN Customer_Status='Churned' THEN 1 ELSE 0 END) / COUNT(*), 2) as Churn_Rate_Pct
FROM customers
GROUP BY Country
HAVING Churn_Rate_Pct > (
    SELECT 100.0 * SUM(CASE WHEN Customer_Status='Churned' THEN 1 ELSE 0 END) / COUNT(*)
    FROM customers
)
ORDER BY Churn_Rate_Pct DESC;
"""
print(pd.read_sql(q22, conn))

     Country  Churn_Rate_Pct
0     Canada           28.61
1     France           28.20
2  Singapore           27.52
3        USA           27.32
4      India           27.05


In [25]:
# Q23 (rewritten with a window function - much faster)
q23 = """
SELECT Customer_ID, Subscription_Plan, Monthly_Fee, Plan_Avg_Fee
FROM (
    SELECT Customer_ID, Subscription_Plan, Monthly_Fee,
           AVG(Monthly_Fee) OVER (PARTITION BY Subscription_Plan) as Plan_Avg_Fee
    FROM customers
)
WHERE Monthly_Fee > Plan_Avg_Fee
ORDER BY Monthly_Fee DESC
LIMIT 10;
"""
print(pd.read_sql(q23, conn))

  Customer_ID Subscription_Plan  Monthly_Fee  Plan_Avg_Fee
0  CUST128701        Enterprise       286.33    255.274598
1  CUST118347        Enterprise       286.31    255.274598
2  CUST113800        Enterprise       286.29    255.274598
3  CUST102119        Enterprise       286.27    255.274598
4  CUST113472        Enterprise       286.24    255.274598
5  CUST107548        Enterprise       286.21    255.274598
6  CUST128295        Enterprise       286.20    255.274598
7  CUST103107        Enterprise       286.18    255.274598
8  CUST130125        Enterprise       286.18    255.274598
9  CUST133032        Enterprise       286.17    255.274598


In [26]:
# Q24: The single most valuable customer (highest Lifetime_Value)
q24 = """
SELECT Customer_ID, Subscription_Plan, Lifetime_Value
FROM customers
WHERE Lifetime_Value = (SELECT MAX(Lifetime_Value) FROM customers);
"""
print(pd.read_sql(q24, conn))

  Customer_ID Subscription_Plan  Lifetime_Value
0  CUST113367        Enterprise        13327.19


In [27]:
# Q25: Customers with fewer support tickets than the average for churned customers
q25 = """
SELECT Customer_ID, Customer_Status, Support_Tickets
FROM customers
WHERE Support_Tickets < (
    SELECT AVG(Support_Tickets) FROM customers WHERE Customer_Status = 'Churned'
)
AND Customer_Status = 'Active'
LIMIT 10;
"""
print(pd.read_sql(q25, conn))

  Customer_ID Customer_Status  Support_Tickets
0  CUST100001          Active                3
1  CUST100005          Active                3
2  CUST100006          Active                1
3  CUST100010          Active                3
4  CUST100012          Active                1
5  CUST100014          Active                0
6  CUST100015          Active                2
7  CUST100017          Active                2
8  CUST100018          Active                3
9  CUST100019          Active                3


In [28]:
# Q26: Using a CTE to first calculate revenue per plan, then filter
q26 = """
WITH plan_revenue AS (
    SELECT Subscription_Plan, SUM(Revenue) as Total_Revenue, COUNT(*) as Customer_Count
    FROM customers
    WHERE Customer_Status = 'Active'
    GROUP BY Subscription_Plan
)
SELECT * FROM plan_revenue
ORDER BY Total_Revenue DESC;
"""
print(pd.read_sql(q26, conn))

  Subscription_Plan  Total_Revenue  Customer_Count
0        Enterprise   907823.44572            3783
1           Premium   623881.56902            6539
2          Standard   357897.05501            8267
3             Basic   100421.51956            6946


In [29]:
# Q27: CTE + JOIN - combine plan revenue with plan metadata
q27 = """
WITH plan_revenue AS (
    SELECT Subscription_Plan, SUM(Revenue) as Total_Revenue
    FROM customers
    WHERE Customer_Status = 'Active'
    GROUP BY Subscription_Plan
)
SELECT pr.Subscription_Plan, pr.Total_Revenue, pd.Support_Level
FROM plan_revenue pr
JOIN plan_details pd ON pr.Subscription_Plan = pd.Subscription_Plan
ORDER BY pr.Total_Revenue DESC;
"""
print(pd.read_sql(q27, conn))

  Subscription_Plan  Total_Revenue              Support_Level
0        Enterprise   907823.44572  Dedicated Account Manager
1           Premium   623881.56902           Priority Support
2          Standard   357897.05501               Email + Chat
3             Basic   100421.51956                 Email Only


In [30]:
# Q28: Multiple CTEs - compare high-value vs low-value customer segments
q28 = """
WITH high_value AS (
    SELECT COUNT(*) as Count, AVG(Customer_Satisfaction) as Avg_Satisfaction
    FROM customers WHERE Lifetime_Value >= 2000
),
low_value AS (
    SELECT COUNT(*) as Count, AVG(Customer_Satisfaction) as Avg_Satisfaction
    FROM customers WHERE Lifetime_Value < 2000
)
SELECT 'High Value' as Segment, Count, ROUND(Avg_Satisfaction,2) as Avg_Satisfaction FROM high_value
UNION ALL
SELECT 'Low Value' as Segment, Count, ROUND(Avg_Satisfaction,2) as Avg_Satisfaction FROM low_value;
"""
print(pd.read_sql(q28, conn))

      Segment  Count  Avg_Satisfaction
0  High Value   7472              7.37
1   Low Value  27528              6.53


In [31]:
# Q29: CTE to find each country's top-paying customer
q29 = """
WITH ranked_customers AS (
    SELECT Customer_ID, Country, Monthly_Fee,
           ROW_NUMBER() OVER (PARTITION BY Country ORDER BY Monthly_Fee DESC) as rn
    FROM customers
)
SELECT Country, Customer_ID, Monthly_Fee
FROM ranked_customers
WHERE rn = 1
ORDER BY Monthly_Fee DESC;
"""
print(pd.read_sql(q29, conn))

     Country Customer_ID  Monthly_Fee
0        UAE  CUST128701       286.33
1        USA  CUST118347       286.31
2  Singapore  CUST113800       286.29
3    Germany  CUST102119       286.27
4         UK  CUST107548       286.21
5     Canada  CUST103107       286.18
6  Australia  CUST118019       286.12
7      India  CUST128671       286.05
8     Brazil  CUST101468       285.85
9     France  CUST123867       285.31


In [32]:
# Q30: CTE calculating churn rate by industry, filtered to only industries with >1000 customers
q30 = """
WITH industry_stats AS (
    SELECT Industry,
           COUNT(*) as Total,
           SUM(CASE WHEN Customer_Status='Churned' THEN 1 ELSE 0 END) as Churned,
           ROUND(100.0 * SUM(CASE WHEN Customer_Status='Churned' THEN 1 ELSE 0 END) / COUNT(*), 2) as Churn_Rate
    FROM customers
    GROUP BY Industry
)
SELECT * FROM industry_stats
WHERE Total > 1000
ORDER BY Churn_Rate DESC;
"""
print(pd.read_sql(q30, conn))

          Industry  Total  Churned  Churn_Rate
0       E-commerce   4369     1212       27.74
1        Education   4253     1179       27.72
2    Manufacturing   4339     1195       27.54
3          Finance   4376     1203       27.49
4       Technology   4420     1179       26.67
5  Marketing/Media   4302     1146       26.64
6       Healthcare   4297     1132       26.34
7           Retail   4294     1130       26.32


In [33]:
# Q31: Rank customers by Lifetime_Value within their own plan
q31 = """
SELECT Customer_ID, Subscription_Plan, Lifetime_Value,
       RANK() OVER (PARTITION BY Subscription_Plan ORDER BY Lifetime_Value DESC) as Rank_In_Plan
FROM customers
WHERE Customer_Status = 'Active'
ORDER BY Subscription_Plan, Rank_In_Plan
LIMIT 15;
"""
print(pd.read_sql(q31, conn))

   Customer_ID Subscription_Plan  Lifetime_Value  Rank_In_Plan
0   CUST117369             Basic          807.46             1
1   CUST120758             Basic          804.08             2
2   CUST122382             Basic          801.49             3
3   CUST127294             Basic          796.50             4
4   CUST122295             Basic          790.21             5
5   CUST118244             Basic          787.61             6
6   CUST118708             Basic          787.24             7
7   CUST112848             Basic          786.96             8
8   CUST134898             Basic          785.05             9
9   CUST112156             Basic          784.17            10
10  CUST121410             Basic          782.89            11
11  CUST129431             Basic          778.23            12
12  CUST134171             Basic          778.09            13
13  CUST112482             Basic          777.72            14
14  CUST130941             Basic          776.65       

In [34]:
# Q32: DENSE_RANK - like RANK but no gaps after ties
q32 = """
SELECT Customer_ID, Subscription_Plan, Monthly_Fee,
       DENSE_RANK() OVER (ORDER BY Monthly_Fee DESC) as Fee_Rank
FROM customers
LIMIT 10;
"""
print(pd.read_sql(q32, conn))

  Customer_ID Subscription_Plan  Monthly_Fee  Fee_Rank
0  CUST128701        Enterprise       286.33         1
1  CUST118347        Enterprise       286.31         2
2  CUST113800        Enterprise       286.29         3
3  CUST102119        Enterprise       286.27         4
4  CUST113472        Enterprise       286.24         5
5  CUST107548        Enterprise       286.21         6
6  CUST128295        Enterprise       286.20         7
7  CUST103107        Enterprise       286.18         8
8  CUST130125        Enterprise       286.18         8
9  CUST133032        Enterprise       286.17         9


In [35]:
# Q33: Running total of revenue, ordered by subscription date (cumulative growth)
q33 = """
SELECT Customer_ID, Subscription_Date, Revenue,
       SUM(Revenue) OVER (ORDER BY Subscription_Date) as Running_Total_Revenue
FROM customers
WHERE Customer_Status = 'Active'
ORDER BY Subscription_Date
LIMIT 15;
"""
print(pd.read_sql(q33, conn))

   Customer_ID    Subscription_Date    Revenue  Running_Total_Revenue
0   CUST106189  2022-01-01 00:00:00   90.74940             1474.66763
1   CUST109982  2022-01-01 00:00:00   46.72175             1474.66763
2   CUST110632  2022-01-01 00:00:00   94.60768             1474.66763
3   CUST111997  2022-01-01 00:00:00   14.22394             1474.66763
4   CUST112219  2022-01-01 00:00:00   38.61205             1474.66763
5   CUST115556  2022-01-01 00:00:00   41.62230             1474.66763
6   CUST116726  2022-01-01 00:00:00  235.89608             1474.66763
7   CUST118708  2022-01-01 00:00:00   16.40216             1474.66763
8   CUST122428  2022-01-01 00:00:00  237.78918             1474.66763
9   CUST125548  2022-01-01 00:00:00   93.85875             1474.66763
10  CUST126357  2022-01-01 00:00:00   13.28100             1474.66763
11  CUST128317  2022-01-01 00:00:00   40.18070             1474.66763
12  CUST132481  2022-01-01 00:00:00  228.40970             1474.66763
13  CUST133104  2022

In [36]:
# Q34: LAG - compare each customer's fee to the previous customer's fee (ordered by signup)
q34 = """
SELECT Customer_ID, Subscription_Date, Monthly_Fee,
       LAG(Monthly_Fee) OVER (ORDER BY Subscription_Date) as Previous_Customer_Fee
FROM customers
ORDER BY Subscription_Date
LIMIT 10;
"""
print(pd.read_sql(q34, conn))

  Customer_ID    Subscription_Date  Monthly_Fee  Previous_Customer_Fee
0  CUST106189  2022-01-01 00:00:00        97.58                    NaN
1  CUST109982  2022-01-01 00:00:00        50.51                  97.58
2  CUST110632  2022-01-01 00:00:00       100.22                  50.51
3  CUST111997  2022-01-01 00:00:00        15.02                 100.22
4  CUST112219  2022-01-01 00:00:00        42.95                  15.02
5  CUST115556  2022-01-01 00:00:00        46.35                  42.95
6  CUST115563  2022-01-01 00:00:00        44.77                  46.35
7  CUST116284  2022-01-01 00:00:00        45.41                  44.77
8  CUST116726  2022-01-01 00:00:00       247.79                  45.41
9  CUST118708  2022-01-01 00:00:00        16.84                 247.79


In [37]:
# Q35: NTILE - split customers into 4 quartiles by Lifetime_Value
q35 = """
SELECT Customer_ID, Lifetime_Value,
       NTILE(4) OVER (ORDER BY Lifetime_Value DESC) as Value_Quartile
FROM customers
LIMIT 20;
"""
print(pd.read_sql(q35, conn))

   Customer_ID  Lifetime_Value  Value_Quartile
0   CUST113367        13327.19               1
1   CUST101936        13172.00               1
2   CUST105294        13165.06               1
3   CUST111450        13038.53               1
4   CUST120011        12944.15               1
5   CUST118206        12926.75               1
6   CUST110345        12633.44               1
7   CUST131254        12577.89               1
8   CUST110569        12564.98               1
9   CUST127473        12541.45               1
10  CUST109448        12541.16               1
11  CUST110596        12510.10               1
12  CUST129810        12505.03               1
13  CUST119406        12500.01               1
14  CUST127402        12488.77               1
15  CUST115364        12462.99               1
16  CUST106091        12456.92               1
17  CUST133032        12402.49               1
18  CUST121298        12392.99               1
19  CUST122571        12381.44               1


In [38]:
# Q36: Percentage of total revenue each plan contributes (window function version)
q36 = """
SELECT Subscription_Plan,
       SUM(Revenue) as Plan_Revenue,
       ROUND(100.0 * SUM(Revenue) / SUM(SUM(Revenue)) OVER (), 2) as Pct_Of_Total_Revenue
FROM customers
WHERE Customer_Status = 'Active'
GROUP BY Subscription_Plan
ORDER BY Pct_Of_Total_Revenue DESC;
"""
print(pd.read_sql(q36, conn))

  Subscription_Plan  Plan_Revenue  Pct_Of_Total_Revenue
0        Enterprise  907823.44572                 45.62
1           Premium  623881.56902                 31.35
2          Standard  357897.05501                 17.98
3             Basic  100421.51956                  5.05


In [39]:
# Q37: Month-over-month change in new signups (using LAG)
q37 = """
WITH monthly_signups AS (
    SELECT strftime('%Y-%m', Subscription_Date) as Month, COUNT(*) as Signups
    FROM customers
    GROUP BY Month
)
SELECT Month, Signups,
       Signups - LAG(Signups) OVER (ORDER BY Month) as Change_From_Prev_Month
FROM monthly_signups
ORDER BY Month
LIMIT 15;
"""
print(pd.read_sql(q37, conn))

      Month  Signups  Change_From_Prev_Month
0   2022-01      713                     NaN
1   2022-02      692                   -21.0
2   2022-03      760                    68.0
3   2022-04      716                   -44.0
4   2022-05      832                   116.0
5   2022-06      722                  -110.0
6   2022-07      746                    24.0
7   2022-08      791                    45.0
8   2022-09      738                   -53.0
9   2022-10      731                    -7.0
10  2022-11      663                   -68.0
11  2022-12      764                   101.0
12  2023-01      741                   -23.0
13  2023-02      643                   -98.0
14  2023-03      715                    72.0


In [40]:
# Q38: Top 3 customers by Lifetime_Value per country
q38 = """
WITH ranked AS (
    SELECT Customer_ID, Country, Lifetime_Value,
           ROW_NUMBER() OVER (PARTITION BY Country ORDER BY Lifetime_Value DESC) as rn
    FROM customers
)
SELECT Country, Customer_ID, Lifetime_Value
FROM ranked
WHERE rn <= 3
ORDER BY Country, rn;
"""
print(pd.read_sql(q38, conn))

      Country Customer_ID  Lifetime_Value
0   Australia  CUST120011        12944.15
1   Australia  CUST110569        12564.98
2   Australia  CUST127473        12541.45
3      Brazil  CUST110143        12207.29
4      Brazil  CUST133726        12039.79
5      Brazil  CUST104111        11994.29
6      Canada  CUST133032        12402.49
7      Canada  CUST119322        12065.78
8      Canada  CUST124418        12032.01
9      France  CUST105294        13165.06
10     France  CUST117039        12157.96
11     France  CUST122433        12153.24
12    Germany  CUST101936        13172.00
13    Germany  CUST104024        12348.30
14    Germany  CUST123367        12184.55
15      India  CUST118206        12926.75
16      India  CUST129810        12505.03
17      India  CUST106091        12456.92
18  Singapore  CUST109448        12541.16
19  Singapore  CUST115364        12462.99
20  Singapore  CUST113927        12173.26
21        UAE  CUST130125        12223.68
22        UAE  CUST112844        1

In [41]:
# Q39: Cumulative percentage of customers reached, ordered by Lifetime_Value (useful for Pareto/80-20 analysis)
q39 = """
WITH ranked AS (
    SELECT Customer_ID, Lifetime_Value,
           ROW_NUMBER() OVER (ORDER BY Lifetime_Value DESC) as rn,
           COUNT(*) OVER () as total_customers
    FROM customers
)
SELECT Customer_ID, Lifetime_Value, rn,
       ROUND(100.0 * rn / total_customers, 2) as Cumulative_Pct_Of_Customers
FROM ranked
LIMIT 10;
"""
print(pd.read_sql(q39, conn))

  Customer_ID  Lifetime_Value  rn  Cumulative_Pct_Of_Customers
0  CUST113367        13327.19   1                         0.00
1  CUST101936        13172.00   2                         0.01
2  CUST105294        13165.06   3                         0.01
3  CUST111450        13038.53   4                         0.01
4  CUST120011        12944.15   5                         0.01
5  CUST118206        12926.75   6                         0.02
6  CUST110345        12633.44   7                         0.02
7  CUST131254        12577.89   8                         0.02
8  CUST110569        12564.98   9                         0.03
9  CUST127473        12541.45  10                         0.03


In [42]:
# Q40: Final summary query - full KPI snapshot in one query
q40 = """
SELECT
    COUNT(*) as Total_Customers,
    SUM(CASE WHEN Customer_Status='Active' THEN 1 ELSE 0 END) as Active_Customers,
    SUM(CASE WHEN Customer_Status='Churned' THEN 1 ELSE 0 END) as Churned_Customers,
    ROUND(100.0 * SUM(CASE WHEN Customer_Status='Churned' THEN 1 ELSE 0 END) / COUNT(*), 2) as Churn_Rate_Pct,
    ROUND(SUM(CASE WHEN Customer_Status='Active' THEN Revenue ELSE 0 END), 2) as MRR,
    ROUND(AVG(CASE WHEN Customer_Status='Active' THEN Revenue END), 2) as ARPU,
    ROUND(AVG(Lifetime_Value), 2) as Avg_CLTV
FROM customers;
"""
print(pd.read_sql(q40, conn))

   Total_Customers  Active_Customers  Churned_Customers  Churn_Rate_Pct  \
0            35000             25535               9465           27.04   

          MRR   ARPU  Avg_CLTV  
0  1990023.59  77.93   1467.62  


In [43]:
queries = {
    "01_10_basic_select_groupby": [q1, q2, q3, q4, q5, q6, q7, q8, q9, q10],
    "11_20_case_joins": [q11, q12, q13, q14, q15, q16, q17, q18, q19, q20],
    "21_30_subqueries_ctes": [q21, q22, q23, q24, q25, q26, q27, q28, q29, q30],
    "31_40_window_functions": [q31, q32, q33, q34, q35, q36, q37, q38, q39, q40],
}

for filename, query_list in queries.items():
    with open(f"../sql/{filename}.sql", "w") as f:
        for i, q in enumerate(query_list, 1):
            f.write(f"-- Query {i}\n{q.strip()}\n\n")

print("All SQL files saved to sql/ folder")

All SQL files saved to sql/ folder
